# Home Credit Default Risk

## Description of the datasets

#### application_{train|test}.csv
This is the main table, broken into two files for Train (with TARGET) and Test (without TARGET).
Static data for all applications. One row represents one loan in our data sample.

#### bureau.csv
All client's previous credits provided by other financial institutions that were reported to Credit Bureau (for clients who have a loan in our sample).
For every loan in our sample, there are as many rows as number of credits the client had in Credit Bureau before the application date.

#### bureau_balance.csv
Monthly balances of previous credits in Credit Bureau.
This table has one row for each month of history of every previous credit reported to Credit Bureau – i.e the table has (#loans in sample * # of relative previous credits * # of months where we have some history observable for the previous credits) rows.

#### POS_CASH_balance.csv
Monthly balance snapshots of previous POS (point of sales) and cash loans that the applicant had with Home Credit.
This table has one row for each month of history of every previous credit in Home Credit (consumer credit and cash loans) related to loans in our sample – i.e. the table has (#loans in sample * # of relative previous credits * # of months in which we have some history observable for the previous credits) rows.

#### credit_card_balance.csv
Monthly balance snapshots of previous credit cards that the applicant has with Home Credit.
This table has one row for each month of history of every previous credit in Home Credit (consumer credit and cash loans) related to loans in our sample – i.e. the table has (#loans in sample * # of relative previous credit cards * # of months where we have some history observable for the previous credit card) rows.

#### previous_application.csv
All previous applications for Home Credit loans of clients who have loans in our sample.
There is one row for each previous application related to loans in our data sample.

#### installments_payments.csv
Repayment history for the previously disbursed credits in Home Credit related to the loans in our sample.
There is a) one row for every payment that was made plus b) one row each for missed payment.
One row is equivalent to one payment of one installment OR one installment corresponding to one payment of one previous Home Credit credit related to loans in our sample.

#### HomeCredit_columns_description.csv
This file contains descriptions for the columns in the various data files.

## Imports

In [2]:
import duckdb

# Establish a connection to an in-memory DuckDB instance
con = duckdb.connect(database=':memory:')

# Write the SQL query to inspect the data directly from the CSV
query = """
    SELECT 
        SK_ID_CURR, 
        TARGET, 
        NAME_CONTRACT_TYPE, 
        CODE_GENDER, 
        AMT_INCOME_TOTAL
    FROM read_csv_auto('data/raw/application_train.csv')
    LIMIT 5;
"""

# Execute the query and fetch the result as a pandas DataFrame
df_sample = con.execute(query).df()

# Display the result
display(df_sample)

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,AMT_INCOME_TOTAL
0,100002,1,Cash loans,M,202500.0
1,100003,0,Cash loans,F,270000.0
2,100004,0,Revolving loans,M,67500.0
3,100006,0,Cash loans,F,135000.0
4,100007,0,Cash loans,M,121500.0


In [13]:
# Register files as Views in DuckDB for easier querying later on
con.execute("CREATE VIEW IF NOT EXISTS app_train AS SELECT * FROM read_csv_auto('data/raw/application_train.csv')")
con.execute("CREATE VIEW IF NOT EXISTS app_test AS SELECT * FROM read_csv_auto('data/raw/application_test.csv')")
con.execute("CREATE VIEW IF NOT EXISTS bureau AS SELECT * FROM read_csv_auto('data/raw/bureau.csv')")
con.execute("CREATE VIEW IF NOT EXISTS bureau_balance AS SELECT * FROM read_csv_auto('data/raw/bureau_balance.csv')")
con.execute("CREATE VIEW IF NOT EXISTS credit_card AS SELECT * FROM read_csv_auto('data/raw/credit_card_balance.csv')")
con.execute("CREATE VIEW IF NOT EXISTS pos_cash AS SELECT * FROM read_csv_auto('data/raw/POS_CASH_balance.csv')")
con.execute("CREATE VIEW IF NOT EXISTS install_payments AS SELECT * FROM read_csv_auto('data/raw/installments_payments.csv')")
con.execute("CREATE VIEW IF NOT EXISTS prev_app AS SELECT * FROM read_csv_auto('data/raw/previous_application.csv')")
con.execute("CREATE VIEW IF NOT EXISTS sample_sub AS SELECT * FROM read_csv_auto('data/raw/sample_submission.csv')")

# SQL Queries

In [14]:
# See what types of values a column has
con.execute("SELECT DISTINCT NAME_EDUCATION_TYPE FROM app_train;").df()

,NAME_EDUCATION_TYPE
0,Incomplete higher
1,Secondary / secondary special
2,Higher education
3,Lower secondary
4,Academic degree


In [17]:
# See what types of values a column has and their counts
query_edu = """
SELECT 
    NAME_EDUCATION_TYPE, 
    COUNT(*) AS count,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM app_train), 2) AS percentage
FROM app_train
GROUP BY NAME_EDUCATION_TYPE
ORDER BY count DESC;
"""
display(con.execute(query_edu).df())

,NAME_EDUCATION_TYPE,count,percentage
0,Secondary / secondary special,218391,71.02
1,Higher education,74863,24.34
2,Incomplete higher,10277,3.34
3,Lower secondary,3816,1.24
4,Academic degree,164,0.05


In [ ]:
# Describe the structure of the CSV
schema_query = """
    DESCRIBE SELECT * FROM app_train;
"""

# Fetch and display schema
df_schema = con.execute(schema_query).df()
display(df_schema)

,column_name,column_type,null,key,default,extra
0,SK_ID_CURR,BIGINT,YES,None,None,None
1,TARGET,BIGINT,YES,None,None,None
2,NAME_CONTRACT_TYPE,VARCHAR,YES,None,None,None
3,CODE_GENDER,VARCHAR,YES,None,None,None
4,FLAG_OWN_CAR,VARCHAR,YES,None,None,None
...,...,...,...,...,...,...
117,AMT_REQ_CREDIT_BUREAU_DAY,DOUBLE,YES,None,None,None
118,AMT_REQ_CREDIT_BUREAU_WEEK,DOUBLE,YES,None,None,None
119,AMT_REQ_CREDIT_BUREAU_MON,DOUBLE,YES,None,None,None
120,AMT_REQ_CREDIT_BUREAU_QRT,DOUBLE,YES,None,None,None


In [5]:
# Filter the df by column name
display(df_schema[df_schema['column_name'] == 'AMT_INCOME_TOTAL'])

,column_name,column_type,null,key,default,extra
7,AMT_INCOME_TOTAL,DOUBLE,YES,None,None,None


In [6]:
# Count total rows and missing values for specific columns
null_query = """
    SELECT 
        COUNT(*) AS total_applicants,
        COUNT(*) - COUNT(AMT_INCOME_TOTAL) AS missing_income,
        COUNT(*) - COUNT(EXT_SOURCE_1) AS missing_ext_source_1,
        COUNT(*) - COUNT(EXT_SOURCE_2) AS missing_ext_source_2,
        COUNT(*) - COUNT(EXT_SOURCE_3) AS missing_ext_source_3
    FROM read_csv_auto('data/raw/application_train.csv');
"""

# Execute and display the result
df_nulls = con.execute(null_query).df()
display(df_nulls)

,total_applicants,missing_income,missing_ext_source_1,missing_ext_source_2,missing_ext_source_3
0,307511,0,173378,660,60965


In [ ]:
# Calculate default rates based on missing data
missingness_query = """
    SELECT
        CASE 
            WHEN EXT_SOURCE_1 IS NULL THEN 'Missing Score'
            ELSE 'Has Score'
        END AS ext_source_1_status,
        COUNT(*) AS total_applicants,
        AVG(TARGET) AS default_rate
    FROM read_csv_auto('data/raw/application_train.csv')
    GROUP BY 
        ext_source_1_status;
"""

# Execute and display the result
df_missingness = con.execute(missingness_query).df()
display(df_missingness)

,ext_source_1_status,total_applicants,default_rate
0,Has Score,134133,0.074955
1,Missing Score,173378,0.085195


DAYS EMPLOYED COL

In [8]:
min_max_query = """
SELECT MIN(DAYS_EMPLOYED), MAX(DAYS_EMPLOYED) 
FROM read_csv_auto('data/raw/application_train.csv');
"""

df_min_max = con.execute(min_max_query).df()
display(df_min_max)

,min(DAYS_EMPLOYED),max(DAYS_EMPLOYED)
0,-17912,365243


In [9]:
income_type_query = """
    SELECT NAME_INCOME_TYPE, COUNT(*)
    FROM read_csv_auto('data/raw/application_train.csv')
    WHERE DAYS_EMPLOYED = 365243
    GROUP BY NAME_INCOME_TYPE;
    """
df_income_type = con.execute(income_type_query).df()
display(df_income_type)

,NAME_INCOME_TYPE,count_star()
0,Pensioner,55352
1,Unemployed,22


In [10]:
impact_query = """
SELECT 
    CASE WHEN DAYS_EMPLOYED = 365243 THEN 'Anomalous' ELSE 'Regular' END AS status,
    COUNT(*) AS total_clients,
    AVG(TARGET) AS default_rate
FROM read_csv_auto('data/raw/application_train.csv')
GROUP BY status;
"""

df_impact = con.execute(impact_query).df()
display(df_impact)

,status,total_clients,default_rate
0,Regular,252137,0.086600
1,Anomalous,55374,0.053996


In [11]:
# Query to check the cleaning of DAYS_EMPLOYED
query_days_clean = """
    SELECT 
        SK_ID_CURR,
        DAYS_EMPLOYED,
        CASE
            WHEN DAYS_EMPLOYED = 365243 THEN NULL 
            ELSE DAYS_EMPLOYED 
        END AS DAYS_EMPLOYED_CLEAN,
        CASE 
            WHEN DAYS_EMPLOYED = 365243 THEN TRUE 
            ELSE FALSE 
        END AS IS_DAYS_EMPLOYED_ANOM
    FROM read_csv_auto('data/raw/application_train.csv')
    LIMIT 10;
"""

df_days_check = con.execute(query_days_clean).df()
display(df_days_check)

,SK_ID_CURR,DAYS_EMPLOYED,DAYS_EMPLOYED_CLEAN,IS_DAYS_EMPLOYED_ANOM
0,100002,-637,-637,False
1,100003,-1188,-1188,False
2,100004,-225,-225,False
3,100006,-3039,-3039,False
4,100007,-3038,-3038,False
5,100008,-1588,-1588,False
6,100009,-3130,-3130,False
7,100010,-449,-449,False
8,100011,365243,<NA>,True
9,100012,-2019,-2019,False


In [12]:
verification_query = """
SELECT 
    NAME_INCOME_TYPE, 
    COUNT(*) AS total,
    AVG(CASE WHEN DAYS_EMPLOYED = 365243 THEN 1 ELSE 0 END) AS ratio_anomali
FROM read_csv_auto('data/raw/application_train.csv')
GROUP BY NAME_INCOME_TYPE;
"""

df_verification = con.execute(verification_query).df()
display(df_verification)

,NAME_INCOME_TYPE,total,ratio_anomali
0,Working,158774,0.000000
1,Student,18,0.000000
2,Maternity leave,5,0.000000
3,State servant,21703,0.000000
4,Unemployed,22,1.000000
5,Pensioner,55362,0.999819
6,Commercial associate,71617,0.000000
7,Businessman,10,0.000000
